### This notebook shows you how to pick up any model present in the huggingface repo and generate CIFs from it.
> In development

In [ ]:
import __init__
import pandas as pd

In [ ]:
# Login to Hugging Face Hub
from huggingface_hub import login, HfApi
import os
from _utils import load_api_keys

API_KEY_PATH = "API_keys.jsonc"
data = load_api_keys(API_KEY_PATH)
hf_key_json = str(data['HF_key'])
login(token=hf_key_json)


In [ ]:
# Explicit Z and Spacegroup generation
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_base" \
    --reduced_formula_list "TiO2" \
    --z_list "2" \
    --spacegroups "P4_2/mnm" \
    --level level_4 \
    --num_return_sequences 5 \
    --max_return_attempts 2 \
    --output_parquet generated_structures.parquet

In [ ]:
# Multiple parallel structures
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_bandgap" \
    --reduced_formula_list "TiO2,SiO2" \
    --z_list "2,4" \
    --condition_lists "1.8,0.0" "5.0,0.0" \
    --level level_3 \
    --num_return_sequences 5 \
    --output_parquet semiconductors.parquet

In [ ]:
# Search through Z=1 to 4 to find valid structures
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_density" \
    --reduced_formula_list "SiO2" \
    --search_zs \
    --condition_lists "2.143,0.0" \
    --level level_2 \
    --num_return_sequences 5 \
    --target_valid_cifs 1 \
    --output_parquet high_density_materials.parquet

In [ ]:
# Level 1 generation unconditionally with property
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_SLME" \
    --condition_lists "25.0" \
    --level level_1 \
    --num_return_sequences 5 \
    --output_parquet solar_screening.parquet

**NEW**: pass pre-picked peaks via CSVs, XY, DAT, or TXT files of the xrd spectra to the script, which processes them so you can conditionally generate easily from XRD. By default, it assumes CuKa wavelength (1.54056).

In [ ]:
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_COD-XRD" \
    --reduced_formula_list "TiO2" \
    --z_list "2" \
    --xrd_files "tests/fixtures/test_rutile_processed.csv" \
    --num_return_sequences 5 \
    --output_parquet xrd_2_struct.parquet

In [ ]:
# Early-stopping Z-Search for XRD
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_COD-XRD" \
    --reduced_formula_list "TiO2" \
    --search_zs \
    --xrd_files "tests/fixtures/test_rutile_processed.csv" \
    --num_return_sequences 5 \
    --max_return_attempts 2 \
    --target_valid_cifs 1 \
    --scoring_mode "none" \
    --output_cif_dir xrd_2_struct/

In [ ]:
# If providing data with a different wavelength (e.g. MoKa), provide --xrd_wavelength
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_Mattergen-XRD" \
    --reduced_formula_list "TiO2" \
    --z_list "2" \
    --xrd_files "tests/fixtures/test_rutile_raw.xy" \
    --xrd_wavelength 0.71073 \
    --num_return_sequences 5 \
    --max_return_attempts 2 \
    --output_cif_dir xrd_2_struct/

Generate across all Z's and rank by LOGP to find the most theoretically confident structure

In [ ]:
!python _load_and_generate.py \
    --hf_model_path "c-bone/CrystaLLM-pi_Mattergen-XRD" \
    --reduced_formula_list "TiO2" \
    --search_zs \
    --xrd_files "tests/fixtures/test_rutile_processed.csv" \
    --num_return_sequences 5 \
    --max_return_attempts 2 \
    --spacegroups "P4_2/mnm" \
    --level "level_4" \
    --target_valid_cifs 3 \
    --temperature 1.0 \
    --scoring_mode "LOGP" \
    --output_cif_dir xrd_2_struct/